# Bronze Ingestion: Postgres -> Landing -> Bronze

Incremental JDBC extraction from the source Postgres OLTP database,
landing raw files, then merging into Bronze Delta tables.

**Pattern per table:**
1. Read last watermark (max `updated_at` already processed) from control table
2. JDBC read only rows with `updated_at > watermark`
3. Write raw extract to Landing volume (audit trail)
4. MERGE into Bronze Delta table (upsert on primary key)
5. Update watermark control table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import datetime

## Configuration

In [0]:
# Secrets are stored in Azure Key Vault, linked to Databricks via the
# "retailflow_scope" secret scope (same scope used for the SP storage credentials).
# Added directly in the Key Vault: pg-host, pg-port, pg-database, pg-user, pg-password

SECRET_SCOPE = "retailflow_scope"

PG_HOST = dbutils.secrets.get(scope=SECRET_SCOPE, key="pg-host")
PG_PORT = dbutils.secrets.get(scope=SECRET_SCOPE, key="pg-port")
PG_DATABASE = dbutils.secrets.get(scope=SECRET_SCOPE, key="pg-database")
PG_USER = dbutils.secrets.get(scope=SECRET_SCOPE, key="pg-user")
PG_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="pg-password")

JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"

CATALOG = "retailflow"
LANDING_VOLUME = f"/Volumes/{CATALOG}/landing/raw_files"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
CONTROL_TABLE = f"{CATALOG}.bronze._ingestion_control"

# Tables to ingest: (source_table, primary_key_col, watermark_col)
TABLES_CONFIG = [
    ("customers", "customer_id", "updated_at"),
    ("products", "product_id", "updated_at"),
    ("orders", "order_id", "updated_at"),
    ("order_items", "order_item_id", "created_at"),  # order_items are insert-only, no updates
    ("payments", "payment_id", "updated_at"),
]

## Setup: catalog, schemas, control table (run once)

In [0]:
STORAGE_ACCOUNT = "datalakealli"
CONTAINER = "retailflow-lake"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {CATALOG}
MANAGED LOCATION '{BASE_PATH}/catalog-root'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.landing
MANAGED LOCATION '{BASE_PATH}/landing'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze
MANAGED LOCATION '{BASE_PATH}/bronze'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver
MANAGED LOCATION '{BASE_PATH}/silver'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold
MANAGED LOCATION '{BASE_PATH}/gold'
""")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOG}.landing.raw_files
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    table_name STRING,
    last_watermark TIMESTAMP,
    last_run_at TIMESTAMP
) USING DELTA
""")

## Helper functions

In [0]:
def get_last_watermark(table_name: str):
    """Fetch the last processed watermark for a table, or a far-past default on first run."""
    row = spark.sql(f"""
        SELECT last_watermark FROM {CONTROL_TABLE}
        WHERE table_name = '{table_name}'
    """).collect()
    if row:
        return row[0]["last_watermark"]
    return datetime.datetime(1900, 1, 1)


def update_watermark(table_name: str, new_watermark):
    """Upsert the watermark for a table after a successful run."""
    spark.sql(f"""
        MERGE INTO {CONTROL_TABLE} AS target
        USING (SELECT '{table_name}' AS table_name,
                      TIMESTAMP('{new_watermark}') AS last_watermark,
                      current_timestamp() AS last_run_at) AS source
        ON target.table_name = source.table_name
        WHEN MATCHED THEN UPDATE SET
            target.last_watermark = source.last_watermark,
            target.last_run_at = source.last_run_at
        WHEN NOT MATCHED THEN INSERT (table_name, last_watermark, last_run_at)
            VALUES (source.table_name, source.last_watermark, source.last_run_at)
    """)


def extract_incremental(source_table: str, watermark_col: str, last_watermark):
    """JDBC incremental read: only rows updated since the last watermark."""
    query = f"""
        (SELECT * FROM retail.{source_table}
         WHERE {watermark_col} > TIMESTAMP '{last_watermark}') AS t
    """
    df = (
        spark.read.format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", query)
        .option("user", PG_USER)
        .option("password", PG_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .option("numPartitions", 4)
        .load()
    )
    return df


def land_and_merge(source_table: str, pk_col: str, watermark_col: str):
    last_watermark = get_last_watermark(source_table)
    print(f"[{source_table}] extracting rows where {watermark_col} > {last_watermark}")

    df = extract_incremental(source_table, watermark_col, last_watermark)
    row_count = df.count()

    if row_count == 0:
        print(f"[{source_table}] no new rows, skipping.")
        return

    df = (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_system", F.lit("postgres"))
    )

    landing_path = f"{LANDING_VOLUME}/{source_table}/dt={datetime.date.today().isoformat()}"
    df.write.mode("append").parquet(landing_path)
    print(f"[{source_table}] landed {row_count} rows to {landing_path}")

    bronze_table = f"{BRONZE_SCHEMA}.{source_table}"

    if not spark.catalog.tableExists(bronze_table):
        df.write.format("delta").saveAsTable(bronze_table)
        print(f"[{source_table}] created Bronze table {bronze_table} with {row_count} rows")
    else:
        delta_table = DeltaTable.forName(spark, bronze_table)
        (
            delta_table.alias("target")
            .merge(df.alias("source"), f"target.{pk_col} = source.{pk_col}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"[{source_table}] merged {row_count} rows into {bronze_table}")

    new_watermark = df.agg(F.max(watermark_col)).collect()[0][0]
    update_watermark(source_table, new_watermark)
    print(f"[{source_table}] watermark advanced to {new_watermark}")

## Run ingestion for all tables

In [0]:
for source_table, pk_col, watermark_col in TABLES_CONFIG:
    try:
        land_and_merge(source_table, pk_col, watermark_col)
    except Exception as e:
        print(f"[{source_table}] FAILED: {e}")
        raise

## Quick validation

In [0]:
for source_table, _, _ in TABLES_CONFIG:
    count = spark.table(f"{BRONZE_SCHEMA}.{source_table}").count()
    print(f"{BRONZE_SCHEMA}.{source_table}: {count} rows")

In [0]:
for source_table, _, _ in TABLES_CONFIG:
    count = spark.table(f"{BRONZE_SCHEMA}.{source_table}").count()
    print(f"{BRONZE_SCHEMA}.{source_table}: {count} rows")